# Online range evaluation (artifact CSV)

Loads **combo-range probabilities** from `artifacts/online_range_history.csv` (see `runner.py session-split` / ``--range-csv-out``). Joins each row to the Pluribus ``.phh`` via `(session, hand_number)` to attach **cumulative board**, **target hole cards**, and **p1–p6** seats.

**Brier**: preflop rows use 1,326 → **169** collapse; postflop rows use full **1,326** keys. **Expected strength** uses Monte Carlo under the row distribution (faster than enumerating all combos); set `STRENGTH_MC_SAMPLES` or a row cap while iterating.

Run from the repo root with the project virtualenv (`.venv`) as the kernel.


In [4]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path.cwd().resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from utils.eval.online_csv import (
    add_calibration_columns,
    combo_probability_columns,
    enrich_online_range_dataframe,
)


In [5]:
# --- configure ---
CSV_PATH = REPO / "artifacts" / "online_range_history.csv"
PLURIBUS_ROOT = REPO / "pluribus"
STRENGTH_MC_SAMPLES = 96
RNG_SEED = 0
VERBOSE = True  # stdout progress from enrich + calibration
PROGRESS_EVERY_ENRICH = 100
PROGRESS_EVERY_CALIB = 50
# set to None for all rows (664 rows × MC may take a few minutes)
MAX_ROWS: int | None = None


In [6]:
df_raw = pd.read_csv(CSV_PATH)
combo_cols = combo_probability_columns(df_raw)
if MAX_ROWS is not None:
    df_raw = df_raw.iloc[: MAX_ROWS].copy()

df = enrich_online_range_dataframe(
    df_raw,
    PLURIBUS_ROOT,
    verbose=VERBOSE,
    progress_every=PROGRESS_EVERY_ENRICH,
)
rng = np.random.default_rng(RNG_SEED)
df = add_calibration_columns(
    df,
    combo_cols,
    strength_mc_samples=STRENGTH_MC_SAMPLES,
    strength_rng=rng,
    verbose=VERBOSE,
    progress_every=PROGRESS_EVERY_CALIB,
)
df


TypeError: enrich_online_range_dataframe() got an unexpected keyword argument 'verbose'

In [ ]:
display(df.groupby("street", sort=False)["brier"].agg(["mean", "count"]))

post = df[df["community_cards"].str.len() >= 6]
display(
    post.groupby("street", sort=False)[
        ["expected_made_pct", "actual_made_pct", "expected_draw", "actual_draw"]
    ].mean(numeric_only=True)
)
